# Asking the bank's documentation

An employee of Orla Bank, a fictional bank, wants to put a new service into
production. The answer is spread over documents of three departments: the
technical guide says how to deploy, the release standard says who approves,
the compliance checklist says what must be checked first. Some documents
are old versions, and some disagree with each other.

This notebook follows that one question through the system, then asks
"what if ..." until every part of the assignment has appeared. The second
half is a reference: the other scenarios, learning, monitoring and the
evaluation numbers.

## The idea

```
 question
    |
    v
 classifier ---- reads the 8 closest chunks of the whole corpus,
    |            picks the departments (domains) to ask
    |
    +--------------------+--------------------+
    v                    v                    v
 technical agent     business agent     compliance agent
 searches only its   (in parallel)      (in parallel)
 own documents,
 writes claims, each
 citing its chunks
    |                    |                    |
    +--------------------+--------------------+
    v
 conflict check ---- a model finds claims that contradict each other;
    |                code picks the winner by fixed rules
    v
 synthesis --------- a model writes one answer citing [TEC-3] ...;
    |                code checks every cited id and lists the sources
    v
 answer, sources, warnings
```

Four principles carry the design:

1. **Code coordinates, models read text.** The steps above are plain Python
   in `rag_system/orchestrator.py`; a model is used only where text must be
   understood: routing, writing claims, spotting a contradiction, writing
   the answer.
2. **Every sentence has a source.** Agents write claims that cite chunk ids;
   code turns them into citations, and the answer may cite only documents
   the agents read.
3. **The winner of a conflict is decided by code, not chosen by a model.**
   Models do the reading: one finds the claims that contradict each other;
   for two documents of the same kind, a judge marks which other documents
   support each side. Code then applies fixed rules: a policy outranks a
   standard, a standard a guide, a guide a runbook; within one kind, the
   side with more supporting documents wins; a tie has no winner.
4. **"I don't know" is an answer.** An agent that finds nothing relevant
   says so; an answer is never filled with what the documents do not say.

## Starting the system

Everything starts from one object, `Assistant` (`rag_system/app.py`). It
reads `.env` to choose the models, loads and indexes the documents in
`data/` and builds the orchestrator. The command line starts from the same
object: `uv run python -m rag_system "your question"`.

**Without a `.env` file the notebook runs in offline mode.** A stand-in
then composes each answer from the retrieved chunks by template, and no
conflicts are detected; the first line printed below says which mode ran.
The outputs saved in this notebook come from a run with Claude
(`claude-sonnet-5-5`).

To run it with Claude yourself, copy `.env.example` to `.env` in the
repository root and set two values:

```
LLM_PROVIDER=anthropic
ANTHROPIC_API_KEY=sk-ant-...
```

Then restart the kernel and run all cells. Startup fails with the names of
the missing variables if the key is not set.

In [1]:
import os
import sys
import time
from pathlib import Path

# Run from the repository root, so that .env, data/ and the package are found.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["PYDANTIC_AI_NO_BANNER"] = "1"

from IPython.display import Markdown, display

from rag_system.app import Assistant, as_markdown
from rag_system.messages import Classification
from rag_system.orchestrator import Orchestrator

started = time.perf_counter()
assistant = Assistant()  # without .env: the offline stand-in
seconds = time.perf_counter() - started
print(assistant.describe())
print(f"Indexed {len(assistant.documents)} documents in {seconds:.1f} s")

Language model: anthropic (claude-sonnet-5-5)
Embedding model: minishlab/potion-base-8M
Indexed 26 documents in 0.5 s


In [2]:
def show(text):
    """Markdown, shown formatted."""
    display(Markdown(text))


def show_answer(answer):
    """The answer, its sources and any warnings."""
    show(as_markdown(answer))

## One question, step by step

The first test scenario of the assignment:

In [3]:
QUESTION = (
    "What's the process for deploying a new microservice and what "
    "compliance checks are needed?"
)
answer = await assistant.ask(QUESTION)

**Step 1, the classifier.** It searches the whole corpus first and gives
the model the question with the 8 closest chunks and their departments; the
model decides which departments to ask and writes a sub-question for each.
Routing from evidence, not from department descriptions: on the first 12
questions, descriptions found 12 of the 18 departments needed, search 18 of
18.

In [4]:
for sub_query in answer.plan.sub_queries:
    history = " (old versions too)" if sub_query.include_history else ""
    print(f"{sub_query.domain:<11} {sub_query.text}{history}")

technical   What's the process for deploying a new microservice?
compliance  What compliance checks are needed for deploying a new microservice?


**Step 2, the domain agents**, in parallel. Each searches only its own
documents (embedding similarity and keyword search, merged), keeps the 8
best chunks plus up to 2 more that are close to the sub-question, and writes
short claims that cite those chunks. The agents never talk to each other.

In [5]:
for agent_answer in answer.agent_answers:
    print(
        f"{agent_answer.domain}: {agent_answer.status}, read "
        f"{len(agent_answer.chunks_read)} chunks, wrote "
        f"{len(agent_answer.claims)} claims; for example:"
    )
    for claim in agent_answer.claims[:2]:
        cited = ", ".join(c.chunk_id for c in claim.citations)
        print(f"    {claim.text}  <- {cited}")

technical: answered, read 8 chunks, wrote 14 claims; for example:
    Before coding, register the service in the service catalog with an owning team, a short description and the highest data classification it will handle (CMP-1).  <- TEC-1#2
    Create the repository from the platform service template. It already has the CI pipeline, a health endpoint, structured logging and the metrics required by the Observability Standard (TEC-6).  <- TEC-1#2
compliance: answered, read 9 chunks, wrote 20 claims; for example:
    A new service or connector must pass the Pre-Production Compliance Checklist before its first production release.  <- CMP-4#1
    The completed checklist is attached to the release request (Production Release Standard, TEC-3).  <- CMP-4#1


**Step 3, the conflict check.** A model reads all claims and names the
pairs that cannot both be true; code settles each pair (next section).

In [6]:
print(f"{len(answer.conflicts)} conflicts found")
for conflict in answer.conflicts:
    print(f"  {conflict.reason}")

0 conflicts found


**Step 4, the synthesis.** A model writes one answer from the claims and
cites document ids in brackets. Code checks that every cited id belongs to
a claim it was given (otherwise the model is asked again) and builds the
source list from the claims' citations, so no source is written by the
model.

In [7]:
show_answer(answer)

## Deployment process

1. **Register the service** in the service catalog with an owning team, a short description and the highest data classification it will handle, before any coding. [TEC-1]
2. **Create the repository** from the platform service template. It already includes the CI pipeline, a health endpoint, structured logging and the required metrics. [TEC-1] If the template doesn't fit, ask in the Platform Engineering channel and don't fork the pipeline. [TEC-1]
3. **Core banking access:** don't call core banking systems directly. Use an existing connector from the registry or build one. [TEC-1]
4. **Pipeline on every push:** unit tests, SAST and dependency vulnerability scan, image build and signing, then automatic deployment to `dev`. A high or critical scan finding stops the pipeline. [TEC-1]
5. **Promote to `test` and `staging`** with the pipeline button. Staging uses masked copies of production data, and real customer data never leaves production. [TEC-1]
6. **Prepare for production:**
   - Complete the Pre-Production Compliance Checklist and attach it to the release request. [TEC-1][CMP-4]
   - Prepare a rollback plan with the previous image tag and any database migration that must be reversed. [TEC-1]
   - Obtain the approvals required by the Production Release Standard. [TEC-1]
7. **Release** Tuesday to Thursday, 09:00 to 15:00. A release outside this window needs an emergency change. [TEC-1]
8. **After release,** watch the service dashboard for 30 minutes. If the error rate rises above 2%, roll back first and investigate later. [TEC-1]

TEC-1 describes the recommended path. The binding rules are in the Production Release Standard (TEC-3) and the Pre-Production Compliance Checklist (CMP-4). [TEC-1]

## Compliance checks

A new service or connector must pass the Pre-Production Compliance Checklist before its first production release. [CMP-4]

**Data**
- Declare the highest data classification in the service catalog. [CMP-4]
- If the service processes confidential customer data, a DPIA must be completed and signed by the Data Protection Office. [CMP-4]
- If anything goes to the cloud language model, use only the certified AI connector and register the use case. [CMP-4]
- Configure retention of logs and records as in the Data Retention Policy. [CMP-4]

**Security**
- The SAST and dependency scans must show no open high or critical finding. [CMP-4]
- Secrets must be in the vault, and none may be in the code, image or configuration files. [CMP-4]
- APIs must follow the Secure API Standard: authentication, token lifetime, rate limits and masking of personal data in logs. [CMP-4]
  - Service-to-service calls inside the platform use mutual TLS, and calls from other bank systems use OAuth 2 client credentials. [CMP-5]
  - Access tokens expire after 15 minutes, and every API checks the caller's permissions on every request. [CMP-5]

**Traceability**
- The service must write the audit events required by Audit Logging for Automation Runs. [CMP-4]
- Logs, metrics and traces must follow the Observability Standard. [CMP-4]

**Review**
- If the service touches confidential data, the four-eyes review must be done and recorded. [CMP-4]
- The reviewer checks that the change does what the request says and nothing more, that data goes only where the Data Classification Policy and the cloud LLM rules allow, that personal data is masked in logs, and that audit events are written. [CMP-6]
- The four-eyes review is separate from the release approvals, and both are needed. One person may not be both the reviewer and a release approver of the same change. [CMP-6]

## Sign-off and timing
- The service owner signs the checklist and a compliance officer checks it, answering within 3 working days. [CMP-4]
- An incomplete checklist is returned to the owner with the missing items listed, and the release cannot be approved until it comes back complete. [CMP-4]
- If the service uses the cloud LLM, registering the use case needs the purpose and workflow, the data classes sent and pseudonymised fields, and the expected calls per month. The Data Protection Office answers within 10 working days. [CMP-2]
- A typical first release takes about half a day for the catalog entry and repository, 1 to 3 days for the checklist, and 1 to 2 days for approvals. [TEC-1]
- The checklist must be repeated if the service starts handling a higher data class, adds a new external destination, or starts using the cloud language model. [CMP-4]

**Sources**

- [TEC-1] Deploying a New Microservice to the Automation Platform, v1, 2026-02-15, "Before you write code"
- [TEC-1] Deploying a New Microservice to the Automation Platform, v1, 2026-02-15, "The pipeline"
- [TEC-1] Deploying a New Microservice to the Automation Platform, v1, 2026-02-15, "Going to production"
- [TEC-1] Deploying a New Microservice to the Automation Platform, v1, 2026-02-15, "Typical first-release timeline"
- [TEC-1] Deploying a New Microservice to the Automation Platform, v1, 2026-02-15
- [CMP-4] Pre-Production Compliance Checklist for Services, v1, 2025-10-01
- [CMP-4] Pre-Production Compliance Checklist for Services, v1, 2025-10-01, "The checklist"
- [CMP-4] Pre-Production Compliance Checklist for Services, v1, 2025-10-01, "Who signs"
- [CMP-4] Pre-Production Compliance Checklist for Services, v1, 2025-10-01, "When to repeat"
- [CMP-5] Secure API Standard, v1, 2025-08-01, "Authentication"
- [CMP-6] Four-Eyes Review for Sensitive Changes, v1, 2025-07-01, "What the reviewer checks"
- [CMP-6] Four-Eyes Review for Sensitive Changes, v1, 2025-07-01, "Relation to release approvals"
- [CMP-2] Cloud LLM Data Rules: What May Be Sent to Azure OpenAI, v1, 2026-04-01, "Registering a use case"

## What if the documents disagree?

The API troubleshooting runbook keeps debug logs for 30 days; the data
retention policy says 90 days. The agents write both as claims, the
detector pairs them, and the rule decides: a policy outranks a runbook.

In [8]:
answer = await assistant.ask("How long can we keep the troubleshooting logs of an API?")
for conflict in answer.conflicts:
    print(f"{conflict.decision}: {conflict.reason}")
    print(f"  A: {conflict.claim_a.text}")
    print(f"  B: {conflict.claim_b.text}")
print()
show_answer(answer)

kind: CMP-3 (policy) overrules TEC-4 (runbook)
  A: Debug logs, which are the troubleshooting logs of an API, are kept for 30 days and then deleted.
  B: API request logs, including debug logs, are kept for 90 days and then deleted.
kind: CMP-3 (policy) overrules TEC-4 (runbook)
  A: Debug logs, which are the troubleshooting logs of an API, are kept for 30 days and then deleted.
  B: Teams may not keep records shorter or longer than the policy says. A shorter period destroys evidence. A longer one keeps personal data without a reason.



**Answer: keep API troubleshooting logs (debug logs) for 90 days, then delete them.**

**Retention**
- API request logs, including debug logs, are kept for 90 days and then deleted [CMP-3].
- Application logs follow the same 90-day period [CMP-3].
- The Observability Standard says log retention follows the Data Retention Policy (CMP-3) [TEC-6]. The Secure API Standard sets no retention period of its own [CMP-5, CMP-3].
- Teams may not keep records shorter or longer than the policy says. A shorter period destroys evidence, and a longer one keeps personal data without a reason [CMP-3].
- Deletion is automatic. Platform Engineering configures the log stores, and teams do not run their own deletion jobs [CMP-3].
- Records under a legal hold are not deleted until the hold is lifted. Compliance places and lifts holds [CMP-3].
- The 90 days covers the longest regular investigation: monthly reconciliation plus the time to open and close a case [CMP-3].

**Conflict settled**
- A claim from the runbook said debug logs are kept for 30 days and then deleted [TEC-4]. It is overruled. The Data Retention Policy (CMP-3) overrules the runbook (TEC-4), so the 90-day period applies.

**Related rules for debug logs**
- Debug logging for an API may be turned on for at most 24 hours [TEC-4]. This limits how long debug logging is active. It does not limit how long the resulting logs are kept.
- Masking must never be turned off [TEC-4]. It stays on at every log level, including debug [CMP-5].
- Personal data is masked before it is written to any log, trace or error message [CMP-5].
- Request and response bodies are not logged in full. Log only the fields you need, masked [CMP-5].
- Production logs are read in the log viewer, which enforces access rights. They are never copied to laptops, tickets or chat [CMP-5].
- Access to production logs of a service that handles confidential data requires an open incident or a support ticket [CMP-5].

**Sources**

- [CMP-3] Data Retention Policy, v1, 2025-06-01, "Retention periods"
- [CMP-3] Data Retention Policy, v1, 2025-06-01, "Rules"
- [CMP-3] Data Retention Policy, v1, 2025-06-01, "Why these periods"
- [CMP-3] Data Retention Policy, v1, 2025-06-01, "Review"
- [TEC-6] Observability Standard: Metrics, Traces and Logs, v1, 2025-11-01, "Logs"
- [TEC-4] API Performance Troubleshooting Runbook, v1, 2026-05-20, "Step 4: debug logging"
- [CMP-5] Secure API Standard, v1, 2025-08-01, "Personal data in logs"
- [CMP-5] Secure API Standard, v1, 2025-08-01, "Troubleshooting production"

## What if a rule changes tomorrow?

A knowledge update is one `store.add()` of the new document. The new
version names the old one in `supersedes`; the store keeps the old version
for history but searches only the version in force. Here a made-up
version 3 of the release standard asks for three approvers.

In [9]:
from datetime import date

from rag_system.ingestion import Document, chunk_document

RELEASE_QUESTION = "Is one sign-off enough to ship a change to production?"
before = await assistant.ask(RELEASE_QUESTION)

version_3 = Document(
    id="TEC-10",
    title="Production Release Standard",
    domain="technical",
    kind="standard",
    version=3,
    effective_date=date(2026, 11, 1),
    supersedes="TEC-3",
    classification="internal",
    owner="Platform Engineering",
    body="""# Production Release Standard (version 3)

This standard replaces version 2 (TEC-3) from 1 November 2026.

## Approvals

A production release needs **three approvers**: a senior engineer from the
team that owns the service, an engineer from Platform Engineering and an
engineer from Information Security. None of them may be the author.
""",
)
updated_store = assistant.new_store()  # a copy; the rest of the notebook is unchanged
updated_store.add(chunk_document(version_3))
updated = Orchestrator(updated_store, assistant.model, timeout_seconds=60)
after = await updated.answer(RELEASE_QUESTION, Classification.INTERNAL)

for label, result in [("Before", before), ("After", after)]:
    cited = ", ".join(sorted({s.doc_id for s in result.sources}))
    show(f"**{label} the update** (sources: {cited})\n\n{result.text}")

**Before the update** (sources: TEC-1, TEC-3)

No. A production release needs two approvers: a senior engineer from the team that owns the service and an engineer from Platform Engineering [TEC-3]. Neither approver may be the author of the change [TEC-3]. The microservice deployment guide also tells you to get the approvals the Production Release Standard requires before you release [TEC-1].

Two cases add requirements:
- **Confidential data:** A change that touches confidential data needs a separate four-eyes review under CMP-6, in addition to the release approvals [TEC-3].
- **Emergency change:** A release outside the window (Tuesday to Thursday, 09:00 to 15:00) counts as an emergency change. It needs the Platform Engineering lead's approval in addition to the two normal approvers [TEC-3].

For emergency changes, the same two approvers apply, and the on-call Platform Engineering engineer may act as the second one [TEC-3]. The claims don't say whether using the on-call engineer as second approver replaces the Platform Engineering lead's approval or whether the lead is still required. I read the lead's approval as still required, because it is stated as an addition to the two approvers. The owner of the Production Release Standard would need to confirm this.

**After the update** (sources: BUS-2, BUS-7, CMP-4, CMP-6, TEC-1, TEC-10, TEC-9)

**No. One sign-off is not enough to ship a change to production.**

**Release approvals**
- Version 3 of the Production Release Standard requires three approvers for a production release [TEC-10].
- They are a senior engineer from the team that owns the service, an engineer from Platform Engineering and an engineer from Information Security. None of them may be the author [TEC-10].
- Version 3 replaces version 2 (TEC-3) from 1 November 2026 [TEC-10].
- The post-incident review of 31 August 2026 says the standard then required two approvers. The emergency change was released with those two plus the Platform Engineering lead's approval [TEC-9]. This differs from version 3.
- No settled ruling on this difference was supplied. The claims suggest it is a matter of timing, because the review predates version 3's start date, but that is my reading and not a ruling. The owner of the Production Release Standard should confirm which number applies to a given release date.
- The deployment guide only says to obtain the approvals the standard requires, and gives no number [TEC-1].

**Business approval of the change**
- For a minor change, the process owner's approval is enough [BUS-7]. Minor changes are a parameter such as a threshold or e-mail address, a newer version of the same certified template, or a new step that uses an already approved connector [BUS-7].
- A major change needs the approvals required for a new process [BUS-7]. Major changes are a new data source or connector, data of a higher classification, or sending data to the cloud language model for the first time [BUS-7]. When in doubt, treat the change as major [BUS-7].
- Under the new-process rules, public or internal data needs the head of the requesting department [BUS-2]. Confidential customer data also needs the Process Board [BUS-2]. Any data sent to the cloud language model also needs a Data Protection Office sign-off [BUS-2].
- Even after business approval, the release follows the Production Release Standard [BUS-7]. The new version must also run in the sandbox against test data [BUS-7].

**Four-eyes review for sensitive changes**
- A second qualified person must check a sensitive change before it reaches production [CMP-6]. This applies when:
  - a workflow, connector or template that reads or writes confidential data is created or changed;
  - a workflow step starts using the cloud language model;
  - a connector to a new system is created;
  - access rights to confidential data are granted or changed [CMP-6].
- This review is separate from the release approvals, and both are needed [CMP-6]. One person cannot be both the reviewer and a release approver of the same change [CMP-6].
- The reviewer must not be the author and must not report to the author. The reviewer must have completed the platform review training [CMP-6].
- For connectors or code, the reviewer must be a developer. For workflow definitions, the reviewer may be a developer or a process analyst [CMP-6].
- A change cannot be released while its review is open [CMP-6].
- Reviews are answered within 3 working days. An urgent change can be reviewed by the on-call Platform Engineering engineer, if that person meets the reviewer conditions [CMP-6].

**New services or connectors**
- A new service or connector must also pass the pre-production compliance checklist before its first production release. The service owner signs it and a compliance officer checks it [CMP-4].
- An incomplete checklist goes back to the service owner, and the release cannot be approved until it returns complete [CMP-4].

**Exception**
- A rollback to the previous version by the process owner is not a change. It needs no approval and is recorded in the audit store [BUS-7].

## What if people say an answer was not useful?

The learning mechanism is simulated: a user rates every source of an
answer as useful or not (here by whether its document is one the gold set
expects, with 10% of the ratings wrong on purpose). A rating is stored with
the question it was given for and counts only for a similar question
(cosine similarity of at least 0.6), so it cannot leak into unrelated
questions. A chunk rated not useful sinks in the order and loses its
guaranteed place among the 8 best, so the agents stop reading it. (A model
cites only what it uses, so the number of sources moves less than the
chunks the agents read; the gold-set table in the reference counts both.)

In [10]:
import random

from rag_system.learning import ChunkWeights, SimulatedUser

LEARN_QUESTION = (
    "How do I troubleshoot API performance issues while following our "
    "security policies?"
)
EXPECTED = {"TEC-4", "TEC-6", "CMP-5"}

weights = ChunkWeights(assistant.embedder)
learner = Orchestrator(
    assistant.store, assistant.model, weights=weights, timeout_seconds=60
)
user = SimulatedUser(random.Random(42), wrong_share=0.1)


def off_target(result):
    return sorted({s.doc_id for s in result.sources} - EXPECTED)


first = await learner.answer(LEARN_QUESTION, Classification.INTERNAL)
for _ in range(3):
    rated = await learner.answer(LEARN_QUESTION, Classification.INTERNAL)
    weights.apply(LEARN_QUESTION, user.rate(rated, EXPECTED))
last = await learner.answer(LEARN_QUESTION, Classification.INTERNAL)


def read(result):
    return {chunk_id for a in result.agent_answers for chunk_id in a.chunks_read}


learned = weights.for_question(LEARN_QUESTION)
rated = {r.chunk_id for r in weights.history}
rated_down = sorted(c for c in rated if learned.weight(c) < 1)
rated_up = sorted(c for c in rated if learned.weight(c) > 1)
print(f"Ratings given: {len(weights.history)}")
print("Sources from unexpected documents:")
print(f"  before: {off_target(first)}, after: {off_target(last)}")
print("\nChunks rated not useful (weight below 1), read by the agents:")
for chunk_id in rated_down:
    before_read = "read" if chunk_id in read(first) else "-"
    after_read = "read" if chunk_id in read(last) else "-"
    print(
        f"  {chunk_id:<9} weight {learned.weight(chunk_id):.2f}  "
        f"before: {before_read:<5} after: {after_read}"
    )
print(
    f"\nChunks rated useful (weight above 1): {len(rated_up)}, "
    f"read after feedback: {len(set(rated_up) & read(last))}"
)

Ratings given: 29
Sources from unexpected documents:
  before: ['TEC-9'], after: ['TEC-9']

Chunks rated not useful (weight below 1), read by the agents:
  CMP-2#6   weight 0.80  before: read  after: -
  CMP-4#2   weight 0.80  before: read  after: -
  CMP-5#6   weight 0.80  before: read  after: -
  CMP-5#7   weight 0.80  before: read  after: -
  TEC-4#2   weight 0.64  before: read  after: -
  TEC-9#3   weight 0.80  before: read  after: read

Chunks rated useful (weight above 1): 6, read after feedback: 6


## What if no document answers?

Meeting rooms are not in the platform's documentation. The classifier may
ask no department at all, and an agent that is asked should report
"nothing relevant"; either way the answer says so.
(With the offline stand-in the agents answer anyway: its 0.4 similarity bar
is a known gap, see the evaluation below.)

In [11]:
answer = await assistant.ask(
    "Can you tell me which meeting rooms are available next Wednesday for a "
    "workshop with the process owners?"
)
print(f"Departments asked: {[a.domain.value for a in answer.agent_answers]}")
for agent_answer in answer.agent_answers:
    print(f"{agent_answer.domain}: {agent_answer.status}")
show(answer.text)

Departments asked: []


The documentation holds nothing that answers this question.

## What if a part breaks?

An empty question is refused before any search. When one part fails after
the question is accepted, the answer keeps what the others found and says
what is missing. Here the compliance part of the index is down.

In [12]:
try:
    await assistant.ask("   ")
except ValueError as error:
    print(f"Empty question: ValueError({error})")


class ComplianceIndexDown:
    """The real store, except that a search of compliance documents fails."""

    def __init__(self, store):
        self._store = store
        self.model = store.model

    def add(self, chunks):
        self._store.add(chunks)

    def search(self, query, domain, clearance, limit, **options):
        if domain == "compliance":
            raise ConnectionError("compliance index unavailable")
        return self._store.search(query, domain, clearance, limit, **options)


broken = Orchestrator(
    ComplianceIndexDown(assistant.store), assistant.model, timeout_seconds=60
)
answer = await broken.answer(QUESTION, Classification.INTERNAL)
for agent_answer in answer.agent_answers:
    error = f" ({agent_answer.error})" if agent_answer.error else ""
    print(f"{agent_answer.domain}: {agent_answer.status}{error}")
print()
show_answer(answer)

Empty question: ValueError(the question is empty)


technical: answered
compliance: failed (ConnectionError: compliance index unavailable)



## Deploying a new microservice

**1. Register and set up**
- Before coding, register the service in the service catalog. You need an owning team, a short description and the highest data classification the service will handle (CMP-1). [TEC-1]
- Create the repository from the platform service template. It already includes the CI pipeline, a health endpoint, structured logging and the metrics required by the Observability Standard (TEC-6). [TEC-1]
- If the template doesn't fit your case, ask in the Platform Engineering channel. Do not fork the pipeline. [TEC-1]
- A service that calls a core banking system must not call it directly. Use an existing connector from the connector registry, or build one as described in TEC-5. [TEC-1]

**2. Pipeline on every push**
- Every push runs unit tests, a SAST and dependency vulnerability scan, container image build and signing, and automatic deployment to `dev`. [TEC-1]
- A SAST or dependency scan finding rated high or critical stops the pipeline. [TEC-1]

**3. Test and staging**
- Promotion to `test` and `staging` is a button in the pipeline. [TEC-1]
- Staging uses masked copies of production data. Real customer data never leaves production. [TEC-1]

**4. Production**
- Complete the Pre-Production Compliance Checklist (CMP-4) and attach it to the release request. [TEC-1]
- Prepare a rollback plan: the previous image tag and any database migration that must be reversed. [TEC-1]
- Get the approvals required by the Production Release Standard (TEC-3). [TEC-1]
- Release inside the release window, Tuesday to Thursday, 09:00 to 15:00. Releases outside the window need an emergency change. [TEC-1]
- Watch the service dashboard for 30 minutes after release. If the error rate rises above 2%, roll back first and investigate later. [TEC-1]

## Compliance checks in summary
- Service catalog registration with a data classification. [TEC-1]
- Automated SAST and dependency scanning, with high or critical findings blocking the pipeline. [TEC-1]
- The Pre-Production Compliance Checklist (CMP-4), attached to the release request. [TEC-1]
- Approvals under the Production Release Standard (TEC-3). [TEC-1]
- Use of connectors rather than direct calls to core banking systems. [TEC-1]

## Timeline and caveats
- A typical first release takes half a day for the catalog entry and repository, 1 to 3 days for the compliance checklist, and 1 to 2 days for approvals. [TEC-1]
- The guide describes the recommended path. The binding rules are in the Production Release Standard (TEC-3) and the Pre-Production Compliance Checklist (CMP-4). [TEC-1]
- The claims I received do not list the checklist items or the specific approvers. Consult CMP-4 and TEC-3 directly for those.

**Sources**

- [TEC-1] Deploying a New Microservice to the Automation Platform, v1, 2026-02-15, "Before you write code"
- [TEC-1] Deploying a New Microservice to the Automation Platform, v1, 2026-02-15, "The pipeline"
- [TEC-1] Deploying a New Microservice to the Automation Platform, v1, 2026-02-15, "Going to production"
- [TEC-1] Deploying a New Microservice to the Automation Platform, v1, 2026-02-15, "Typical first-release timeline"
- [TEC-1] Deploying a New Microservice to the Automation Platform, v1, 2026-02-15

---

# Reference

## The other two test scenarios

In [13]:
for question in [
    "How do I troubleshoot API performance issues while following our "
    "security policies?",
    "What business approvals are required for implementing a new data "
    "processing workflow?",
]:
    answer = await assistant.ask(question)
    show(f"### {question}")
    show_answer(answer)

### How do I troubleshoot API performance issues while following our security policies?

Follow the API Performance Troubleshooting Runbook and apply the Secure API Standard at each step. No conflicts between the claims needed settling.

**When to use it**
- Use the runbook when a platform API is slower than its objective of p95 latency below 800 ms. The alert fires when p95 stays above 800 ms for 10 minutes [TEC-4].
- The Secure API Standard applies to every API on the platform, internal or exposed [CMP-5]. It covers authentication, token lifetime, rate limits and masking of personal data in logs [CMP-4].

**Steps**
1. **Confirm and scope.**
   - Check p95 and p99 for the last hour on the API's latency dashboard [TEC-4].
   - Check whether all endpoints are slow or only some. One slow endpoint usually means one slow dependency [TEC-4].
   - Check the error rate. If it is above 2%, treat the case as an incident and follow the Platform Incident Response Runbook [TEC-4].
2. **Look for a recent change.**
   - List the releases of the last 24 hours for the API and its dependencies in the change log [TEC-4].
   - If a release matches the start of the slowdown, roll it back and watch the dashboard for 15 minutes [TEC-4].
3. **Check dependencies.**
   - Open the trace view and sort the spans of slow requests by duration [TEC-4].
   - Calls to core banking through RPA connectors are the most common cause. A connector handles one session at a time, so a long queue makes every request wait [TEC-4].
   - Check the database connection pool. At 100% use, requests wait for a connection [TEC-4].
4. **Debug logging, only if traces do not explain the slowdown.**
   - Turn it on for at most 24 hours [TEC-4].
   - Debug logs are kept for 30 days and then deleted [TEC-4].
5. **Escalate.**
   - If the cause is still unknown after one hour, page the Platform Engineering on-call engineer [TEC-4].
   - Attach the dashboard links, the traces you looked at and the list of recent changes [TEC-4].

**Security rules to follow while troubleshooting**
- Debug logs must be masked like all other logs. Never turn off masking to see the raw payload [TEC-4].
- Read production logs only in the log viewer, which enforces access rights. Never copy them to laptops, tickets or chat [CMP-5][TEC-4].
- To access production logs of a service that handles confidential data, you need an open incident or a support ticket [CMP-5].
- Never copy real customer data to test environments. Use the masked copies the platform provides [CMP-5].
- If you send confidential data to the cloud language model outside the rules, it is a data incident. Follow the Platform Incident Response Runbook and notify the Data Protection Office within one hour [CMP-2].
- Exceptions to the Secure API Standard come only from Information Security, in writing, with an expiry date. An expired exception is treated as a finding [CMP-5].

**Example: 31 August 2026 screening connector slowdown**
- The error rate was normal, so requests were slow rather than failing [TEC-9].
- The change log showed no recent release, so rollback was not an option [TEC-9].
- The trace view sorted by span duration showed robot sessions spending about 30 seconds in one step, opening the alert detail screen [TEC-9].

**Sources**

- [TEC-4] API Performance Troubleshooting Runbook, v1, 2026-05-20
- [TEC-4] API Performance Troubleshooting Runbook, v1, 2026-05-20, "Step 1: confirm and scope"
- [TEC-4] API Performance Troubleshooting Runbook, v1, 2026-05-20, "Step 2: look for a recent change"
- [TEC-4] API Performance Troubleshooting Runbook, v1, 2026-05-20, "Step 3: check the dependencies"
- [TEC-4] API Performance Troubleshooting Runbook, v1, 2026-05-20, "Step 4: debug logging"
- [TEC-4] API Performance Troubleshooting Runbook, v1, 2026-05-20, "Step 5: escalate"
- [CMP-5] Secure API Standard, v1, 2025-08-01, "Troubleshooting production"
- [CMP-5] Secure API Standard, v1, 2025-08-01
- [CMP-5] Secure API Standard, v1, 2025-08-01, "Exceptions"
- [CMP-4] Pre-Production Compliance Checklist for Services, v1, 2025-10-01, "The checklist"
- [CMP-2] Cloud LLM Data Rules: What May Be Sent to Azure OpenAI, v1, 2026-04-01, "Breaches"
- [TEC-9] Post-Incident Review: Screening Connector Slowdown, 31 August 2026, v1, 2026-09-07

### What business approvals are required for implementing a new data processing workflow?

**Approvals required depend on the data the workflow uses:**

- **Public or internal data only:** approval from the head of the requesting department is enough [BUS-2].
- **Customer data (confidential under CMP-1):** approval from the head of the requesting department and the Process Board [BUS-2].
- **Any data sent to the cloud language model:** the approvals above plus a sign-off from the Data Protection Office (CMP-2) [BUS-2].

**Version 2 of the policy:** The department head is no longer the only approver. Version 2 added approvals for processes that handle customer data or use the cloud language model [BUS-2]. The rule that the department head alone approves therefore applies only to processes using public or internal data.

**Process Board timing:** The board meets every two weeks. A request must reach the board secretary five working days before a meeting to be on its agenda [BUS-2].

**What the request must contain:**
- The current process step by step, a business case (per BUS-5), the systems touched, the highest data classification, and the process owner [BUS-2].
- The process owner requests the automation and signs the business case [BUS-6].
- The process analyst attaches an assessment covering the process map, volume, proposed templates and connectors, and risks [BUS-4].

**Sources**

- [BUS-2] New Process Approval Policy, v2, 2026-04-01, "Who approves"
- [BUS-2] New Process Approval Policy, v2, 2026-04-01, "What changed from version 1"
- [BUS-2] New Process Approval Policy, v2, 2026-04-01, "The request"
- [BUS-6] Roles: Process Owner, Process Analyst, Developer, v1, 2025-06-01, "Process owner"
- [BUS-4] Process Analyst Handbook, v1, 2026-01-20, "5. Hand-over to the approvers"

## A conflict with no winner

BUS-6 says the process analyst reads the audit records of the workflows
they built; CMP-7 leaves analysts out of the read-access list. Both are
standards, and no other document supports either side, so the answer shows
both and names the owners who must settle it.

In [14]:
answer = await assistant.ask(
    "Can the analyst who built a workflow look at its audit records?"
)
for conflict in answer.conflicts:
    print(f"{conflict.decision}: {conflict.reason}")
print()
show_answer(answer)

no_winner: BUS-6 and CMP-7 are both standards and no other document supports each; ask the owners: Process Excellence Office; Compliance, Internal Audit
no_winner: BUS-6 and CMP-7 are both standards and no other document supports each; ask the owners: Process Excellence Office; Compliance, Internal Audit



**The sources do not settle whether the analyst who built a workflow can look at its audit records. This conflict is unresolved.**

- **Business position:** The process analyst reads the audit records of the workflows they assembled [BUS-6]. The stated purpose is to check the first runs after go-live [BUS-6].
- **Compliance position:** Read access to audit records is limited to Compliance, Internal Audit, the Data Protection Office and, for their own workflows, process owners [CMP-7]. The compliance material does not say that a workflow's builder has read access as such. An analyst could read the records only if they are also the process owner, and the sources do not state whether that is the case [CMP-7].

**Why there is no winner:** BUS-6 and CMP-7 are both standards, and no other document supports either side. No rule can overrule one claim in favor of the other.

**Who must resolve it:** The Process Excellence Office (owner of BUS-6) and Compliance and Internal Audit (owners of CMP-7) must decide.

**Until they do:** The analyst should not assume access based on having built the workflow. Access is confirmed only if the analyst is also the workflow's process owner, and that is not established in the sources [CMP-7].

**Sources**

- [BUS-6] Roles: Process Owner, Process Analyst, Developer, v1, 2025-06-01, "Process analyst"
- [CMP-7] Audit Logging for Automation Runs, v1, 2025-07-01, "Where it is stored"

## A question about the past

Superseded versions stay in the index but are searched only when the
classifier's model marks the question as one about history.

In [15]:
answer = await assistant.ask(
    "How many approvers did a production release need in 2025?"
)
show_answer(answer)

A production release needed one approver in 2025. Version 1 of the Production Release Standard was in force throughout the year, because version 2 replaces it only from 1 February 2026 [TEC-3].

- **Normal approver:** Under version 1, the approver was a senior engineer from the team that owns the service [TEC-2]. The approver could not be the author of the change [TEC-2].
- **Emergency changes:** A release outside the window counted as an emergency change. It needed the Platform Engineering lead's approval in addition to the normal approver [TEC-2]. Such a release therefore needed two approvals.
- **Version 2:** Version 2 raises the requirement to two approvers, a senior engineer from the owning team and an engineer from Platform Engineering [TEC-3]. This applies from 1 February 2026, not to 2025 [TEC-3].

The settled rule is that the version in force on the date of the release applies. Version 2 does not apply to 2025 releases, because it takes effect only on 1 February 2026 [TEC-3].

**Sources**

- [TEC-3] Production Release Standard, v2, 2026-02-01
- [TEC-3] Production Release Standard, v2, 2026-02-01, "Approvals"
- [TEC-3] Production Release Standard, v2, 2026-02-01, "What changed from version 1"
- [TEC-2] Production Release Standard, v1, 2025-01-15, "Approvals"
- [TEC-2] Production Release Standard, v1, 2025-01-15, "Release windows"

## Learning over the gold set

The 19 tuning questions answered before and after three rounds of
simulated feedback. This table always runs on the offline stand-in: it
answers the 19 questions eight times, and the weights act on retrieval,
which is the same code in both modes. "Off-target" counts sources from
documents the question does not expect. The first row is circular (the
simulated user rates by the same expected documents that the row counts);
the paraphrases and the questions that were never rated are the honest
rows.

In [16]:
import yaml

from rag_system.offline_model import offline_model

gold = yaml.safe_load((ROOT / "tests/evals/questions.yaml").read_text())
tuning = [q for q in gold if not q.get("heldout")]


async def found_and_off_target(system, questions, field="question"):
    found = off = 0
    for q in questions:
        clearance = Classification(q.get("clearance", "internal"))
        result = await system.answer(q[field], clearance)
        expected = set(q["expected_documents"])
        found += len(expected & {s.doc_id for s in result.sources})
        off += sum(s.doc_id not in expected for s in result.sources)
    return found, off


async def feedback(system, table_weights, questions):
    rater = SimulatedUser(random.Random(42), wrong_share=0.1)
    for _ in range(3):
        for q in questions:
            clearance = Classification(q.get("clearance", "internal"))
            result = await system.answer(q["question"], clearance)
            ratings = rater.rate(result, set(q["expected_documents"]))
            table_weights.apply(q["question"], ratings)


rows = []
table_weights = ChunkWeights(assistant.embedder)
offline_learner = Orchestrator(assistant.store, offline_model(), weights=table_weights)
same_before = await found_and_off_target(offline_learner, tuning)
para_before = await found_and_off_target(offline_learner, tuning, "paraphrase")
await feedback(offline_learner, table_weights, tuning)
same_after = await found_and_off_target(offline_learner, tuning)
para_after = await found_and_off_target(offline_learner, tuning, "paraphrase")
rows.append(("same questions", same_before, same_after))
rows.append(("paraphrased", para_before, para_after))

table_weights = ChunkWeights(assistant.embedder)
offline_learner = Orchestrator(assistant.store, offline_model(), weights=table_weights)
not_rated = tuning[1::2]
before_nr = await found_and_off_target(offline_learner, not_rated)
await feedback(offline_learner, table_weights, tuning[0::2])
after_nr = await found_and_off_target(offline_learner, not_rated)
rows.append(("never rated", before_nr, after_nr))

print(f"{'':<16}{'found before':>13}{'off-target':>11}", end="")
print(f"{'found after':>13}{'off-target':>11}")
for name, (f0, o0), (f1, o1) in rows:
    print(f"{name:<16}{f0:>13}{o0:>11}{f1:>13}{o1:>11}")

                 found before off-target  found after off-target
same questions             29        210           28         11
paraphrased                28        284           28        182
never rated                14         64           14         64


## Monitoring

Every answer of the orchestrator above was recorded: outcome, duration and,
per agent, its status and time. In production the same records would go to
OpenTelemetry; PydanticAI already instruments the model calls.

In [17]:
print(assistant.monitor.report())

questions: 8, successful: 88%

agent        asked  answered  nothing  failed  seconds  useful
technical        5      100%       0%      0%     4.24       -
business         2      100%       0%      0%     4.50       -
compliance       4      100%       0%      0%     4.58       -


## Evaluation

The gold set (`tests/evals/questions.yaml`) has 19 tuning questions, on
which the thresholds were set, and 35 held-out questions, written later by
other models from the document titles alone and run once with the
thresholds frozen. The offline measures below run here, with the same code
the tests use (`tests/evals/measures.py`).

In [18]:
from tests.evals.measures import (
    classifier_counts,
    document_ranks,
    fact_counts,
    store_counts,
)

heldout = [q for q in gold if q.get("heldout")]
print(f"{'offline measure':<34}{'tuning':>10}{'held-out':>10}")
results = {}
for name, questions in [("tuning", tuning), ("held-out", heldout)]:
    found, extra, _, _ = await classifier_counts(assistant.store, questions)
    expected_domains = sum(len(q["expected_domains"]) for q in questions)
    in_top_5, documents, _ = store_counts(assistant.store, questions)
    ranks = document_ranks(assistant.store, questions)
    facts, all_facts, _ = await fact_counts(assistant.store, questions)
    results[name] = [
        f"{found}/{expected_domains}",
        f"{extra / len(questions):.1f}",
        f"{in_top_5 / documents:.0%}",
        f"{sum(1 / r for r in ranks) / len(ranks):.2f}",
        f"{facts / all_facts:.0%}",
    ]
labels = [
    "classifier: domains found",
    "classifier: extra per question",
    "document in its domain's top 5",
    "mean reciprocal rank (random 0.34)",
    "facts in the chunks kept",
]
for i, label in enumerate(labels):
    print(f"{label:<34}{results['tuning'][i]:>10}{results['held-out'][i]:>10}")

offline measure                       tuning  held-out


classifier: domains found              26/26     38/38
classifier: extra per question           0.7       1.1
document in its domain's top 5           91%       68%
mean reciprocal rank (random 0.34)      0.75      0.59
facts in the chunks kept                 92%       65%


With the language model (`tests/evals/test_llm.py`, claude-sonnet-5-5,
2026-10-30; run with `uv run pytest -m eval` and a key in `.env`):

| measure | tuning | held-out | common target |
|---|---|---|---|
| facts in the chunks the model read (context recall) | 92% | 68-70% | 75-80% |
| facts stated in the answer (judge model) | 88% | 73-76% | |
| statements supported by the sources (faithfulness) | 97% overall | | 90-95% in regulated work |
| questions no document answers, ended as "nothing relevant" | | 6 of 7 | |
| conflicts found and decided as expected | 3-5 of 5 per run | | |

The gap between tuning and held-out is in retrieval: on new wording the
chunk with the answer often ranks below the first 8 of its department.
The local embedding model (potion-base-8M, chosen so the demo runs without
a key) is the main limit; production would use Azure OpenAI embeddings.